In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Activation,Dropout,Conv2D, MaxPooling2D,BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model, Sequential
import numpy as np
import pandas as pd
import shutil
import time
import cv2 as cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import os
import seaborn as sns
sns.set_style('darkgrid')
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from IPython.core.display import display, HTML

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
%matplotlib inline
import tensorflow as tf
import keras
import glob
import cv2
import pickle, datetime

from keras.models import Model, Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D,GlobalAveragePooling2D,Lambda
from keras.layers import LSTM, Input, TimeDistributed,Convolution2D,Activation
from keras.layers import ZeroPadding2D
from keras.optimizers import RMSprop, SGD
from keras.layers import BatchNormalization
from tensorflow.keras.utils import to_categorical
from keras.preprocessing import sequence
from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from keras.models import load_model
from keras import models
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
# Import the backend
from keras import backend as K
import os
from torchvision import models

In [8]:
sdir='/content/drive/MyDrive/BrainUltrasound_Multiclass'
categories=['test', 'train', 'valid']
for category in categories:
    category_path=os.path.join(sdir, category)
    filepaths=[]
    labels=[]
    classlist=os.listdir(category_path)
    for klass in classlist:
        classpath=os.path.join(category_path,klass)
        flist=os.listdir(classpath)
        for f in flist:
            fpath=os.path.join(classpath,f)
            filepaths.append(fpath)
            labels.append(klass)
    Fseries=pd.Series(filepaths, name='filepaths')
    Lseries=pd.Series(labels, name='labels')
    if category=='test':
        test_df=pd.concat([Fseries, Lseries], axis=1)
    elif category=='train':
        train_df=pd.concat([Fseries, Lseries], axis=1)
    else:
        valid_df=pd.concat([Fseries, Lseries], axis=1)

print('train_df length: ', len(train_df), ' test_df length: ', len(test_df), '  valid_df length: ', len(valid_df))
print (train_df['labels'].value_counts())

train_df length:  1285  test_df length:  82   valid_df length:  272
normal                       226
mild-ventriculomegaly        187
moderate-ventriculomegaly    178
arachnoid-cyst               114
encephalocele                113
cerebellah-hypoplasia        111
severe-ventriculomegaly      105
polencephaly                  76
colphocephaly                 72
intracranial-hemorrdge        41
anold-chiari-malformation     33
m-magna                       29
Name: labels, dtype: int64


In [9]:
working_dir='/content'
aug_dir=os.path.join(working_dir, 'aug')
'''if os.path.isdir(aug_dir):
    shutil.rmtree(aug_dir)
os.mkdir(aug_dir)
for label in train_df['labels'].unique():
    dir_path=os.path.join(aug_dir,label)
    os.mkdir(dir_path)
print(os.listdir(aug_dir))'''

"if os.path.isdir(aug_dir):\n    shutil.rmtree(aug_dir)\nos.mkdir(aug_dir)\nfor label in train_df['labels'].unique():\n    dir_path=os.path.join(aug_dir,label)\n    os.mkdir(dir_path)\nprint(os.listdir(aug_dir))"

In [10]:
gen=ImageDataGenerator(horizontal_flip=True, vertical_flip=True, rotation_range=90)
groups=train_df.groupby('labels') # group by class
for label in train_df['labels'].unique():  # for every class
    group=groups.get_group(label)  # a dataframe holding only rows with the specified label
    sample_count=len(group)   # determine how many samples there are in this class
    if sample_count<50:
      target= sample_count*3
    elif (50<=sample_count<100):
      target= sample_count*2
    elif (100<=sample_count<150):
      target= int(sample_count*1.5)
    else:
      target=sample_count

    if sample_count< target: # if the class has less than target number of images
        aug_img_count=0
        delta=target-sample_count  # number of augmented images to create
        target_dir=os.path.join(aug_dir, label)  # define where to write the images
        aug_gen=gen.flow_from_dataframe( group,  x_col='filepaths', y_col=None, target_size=(400,400), class_mode=None, batch_size=1,
                                         shuffle=True, save_to_dir=target_dir, save_prefix='aug-',save_format='jpg')
        while aug_img_count<delta:
            images=next(aug_gen)
            aug_img_count += len(images)

Found 114 validated image filenames.
Found 111 validated image filenames.
Found 72 validated image filenames.
Found 113 validated image filenames.
Found 33 validated image filenames.
Found 29 validated image filenames.
Found 76 validated image filenames.
Found 105 validated image filenames.
Found 41 validated image filenames.


In [11]:
aug_fpaths=[]
aug_labels=[]
classlist=os.listdir(aug_dir)
for klass in classlist:
    classpath=os.path.join(aug_dir, klass)
    flist=os.listdir(classpath)
    for f in flist:
        fpath=os.path.join(classpath,f)
        aug_fpaths.append(fpath)
        aug_labels.append(klass)
Fseries=pd.Series(aug_fpaths, name='filepaths')
Lseries=pd.Series(aug_labels, name='labels')
aug_df=pd.concat([Fseries, Lseries], axis=1)
print ('length of aug_df" ', len(aug_df))
train_df=pd.concat([train_df,aug_df], axis=0).reset_index(drop=True)
train_df=train_df.sample(frac=1.0, replace=False, random_state=123, axis=0).reset_index(drop=True)
print ('length of train_df is: ', len(train_df))
print (train_df['labels'].value_counts())

length of aug_df"  1148
length of train_df is:  2433
polencephaly                 228
arachnoid-cyst               228
normal                       226
encephalocele                225
cerebellah-hypoplasia        221
colphocephaly                216
severe-ventriculomegaly      209
intracranial-hemorrdge       205
mild-ventriculomegaly        187
moderate-ventriculomegaly    178
anold-chiari-malformation    165
m-magna                      145
Name: labels, dtype: int64


In [12]:
train_brain_images = []
train_brain_labels = []
for i in train_df.index:
    img = cv2.imread(train_df['filepaths'][i], cv2.IMREAD_COLOR)
    if img is None:
      # print some err
      continue
    img = cv2.resize(img, (227, 227))
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    train_brain_images.append(img)
    train_brain_labels.append(train_df["labels"][i])
train_brain_images = np.array(train_brain_images)
train_brain_labels = np.array(train_brain_labels)

In [13]:
label_to_id = {v:i for i,v in enumerate(np.unique(train_brain_labels))}
id_to_label = {v: k for k, v in label_to_id.items()}
train_label_ids = np.array([label_to_id[x] for x in train_brain_labels])

In [14]:
train_brain_images.shape, train_label_ids.shape, train_brain_labels.shape

((2432, 227, 227, 3), (2432,), (2432,))

In [15]:
# test
test_brain_images = []
test_brain_labels = []
for directory_path in glob.glob("/content/drive/MyDrive/BrainUltrasound_Multiclass/valid/*"):
    brain_label = directory_path.split("/")[-1]
    for img_path in glob.glob(os.path.join(directory_path, "*.jpg")):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)

        img = cv2.resize(img, (227, 227))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        test_brain_images.append(img)
        test_brain_labels.append(brain_label)
test_brain_images = np.array(test_brain_images)
test_brain_labels = np.array(test_brain_labels)

In [16]:
test_label_ids = np.array([label_to_id[x] for x in test_brain_labels])

In [17]:
test_brain_images.shape, test_label_ids.shape

((271, 227, 227, 3), (271,))

In [18]:
x_train, y_train, x_test, y_test, N_CATEGORY =train_brain_images,train_brain_labels,test_brain_images,test_brain_labels,len(label_to_id)

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape, N_CATEGORY)

(2432, 227, 227, 3) (2432,) (271, 227, 227, 3) (271,) 12


### Classes with its ids

In [19]:
id_to_label

{0: 'anold-chiari-malformation',
 1: 'arachnoid-cyst',
 2: 'cerebellah-hypoplasia',
 3: 'colphocephaly',
 4: 'encephalocele',
 5: 'intracranial-hemorrdge',
 6: 'm-magna',
 7: 'mild-ventriculomegaly',
 8: 'moderate-ventriculomegaly',
 9: 'normal',
 10: 'polencephaly',
 11: 'severe-ventriculomegaly'}

### Creation of AlexNet structure of CNN

In [20]:
#input image of size 227x227x3 (3 for RGB)

model = tf.keras.Sequential([

    tf.keras.layers.Conv2D(96,11,strides=4,padding='valid',activation='relu', input_shape=(227, 227, 3)),
    #input is of size: 55x55x96
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(3,strides=2,padding='valid'),
    #input is of size: 27x27x96

    tf.keras.layers.Conv2D(256,5,strides=1,padding='same',activation='relu'),
    #input is of size: 27x27x256
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(3,strides=2,padding='valid'),
    #input is of size: 13x13x256


    tf.keras.layers.Conv2D(384,3,strides=1,padding='same',activation='relu'),
    #input is of size: 13x13x384
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(384,3,strides=1,padding='same',activation='relu'),
    #input is of size: 13x13x384
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(256,3,strides=1,padding='same',activation='relu'),
    #input is of size: 13x13x256
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(3,strides=2,padding='valid'),
    #input is of size: 6x6x256

    tf.keras.layers.Flatten(),
    #input is of shape(None,9216)

    tf.keras.layers.Dense(4096,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    #4096 neurons
    tf.keras.layers.Dense(4096,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    #4096 neurons
    tf.keras.layers.Dense(12, activation='softmax')
    #16 class probabilities as output
])

In [21]:
model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [22]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 55, 55, 96)        34944     
                                                                 
 batch_normalization (Batch  (None, 55, 55, 96)        384       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 27, 27, 96)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 27, 27, 256)       614656    
                                                                 
 batch_normalization_1 (Bat  (None, 27, 27, 256)       1024      
 chNormalization)                                                
                                                        

In [25]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:

#Normalization of the images and one-hot encoding of the labels
X_normalized = np.array(x_train / 255.0 )
X_normalized_test = np.array(x_test / 255.0 )

label_binarizer = LabelBinarizer()
y_one_hot = label_binarizer.fit_transform(y_train)
y_one_hot_test = label_binarizer.fit_transform(y_test)

'''y_one_hot = np.array(y_one_hot)
y_one_hot_test = np.array(y_one_hot_test)'''

In [24]:
#Training the AlexNet model with the normalized image data and labels
model.fit(X_normalized, y_one_hot, batch_size=4, epochs=10,verbose=1, validation_data=[X_normalized_test,y_one_hot_test])

Epoch 1/10


ValueError: ignored

In [ ]:
'''train_steps_per_epoch = len(X_normalized) // batch_size
val_steps_per_epoch = len(X_normalized_test) // batch_size'''

In [ ]:
'''def data_generator(X, y, batch_size):
    num_samples = X.shape[0]
    while True:
        indices = np.arange(num_samples)
        np.random.shuffle(indices)
        for i in range(0, num_samples, batch_size):
            batch_indices = indices[i:i + batch_size]
            yield X[batch_indices], y[batch_indices]'''

In [ ]:
# train_data_generator = data_generator(X_normalized, y_one_hot, batch_size)
# validation_data_generator = data_generator(X_normalized_test, y_one_hot_test, batch_size)

In [ ]:
#Save the AlexNet model for the future use(as it takes hours to be trained!)
model.save('alexnetfetal.h5')

### Feature Extraction by CNN

In [ ]:
#Pick the first Fully-Connected layer as the features which will be of dimension (1 x 4096)
layer_name = 'dense_1'
FC_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)

In [ ]:
#Find the Features for n number of train images and we will get n x 4096
#This means we will get 4096 features for each images.
i=0
features=np.zeros(shape=(x_train.shape[0],4096))
for directory_path in glob.glob("/content/drive/MyDrive/BrainUltrasound_Multiclass/train/*"):
    for img_path in glob.glob(os.path.join(directory_path, "*.jpg")):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.resize(img, (227, 227))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        img = np.expand_dims(img, axis=0)
        FC_output = FC_layer_model.predict(img)
        features[i]=FC_output
        i+=1


In [ ]:
#Save the features of the train images to use it in future.
np.save('features', features)

In [ ]:
#Name the feature rows as f_0, f_1, f_2...
feature_col=[]
for i in range(4096):
    feature_col.append("f_"+str(i))
    i+=1


In [ ]:
#Create DataFrame with features and coloumn name
train_features=pd.DataFrame(data=features,columns=feature_col)
feature_col = np.array(feature_col)

train_class = list(np.unique(train_label_ids))
print('Training Features Shape:', train_features.shape)
print('Training Labels Shape:', train_label_ids.shape)
train_class

### Random Forest as Classifier

In [ ]:
#Feed the extracted features with the labels to RANDOM FOREST
rf = RandomForestClassifier(n_estimators = 12, random_state = 42,max_features=50)

rf.fit(train_features, train_label_ids)

### Testing the Novel Algorithm

#### Feature Extraction by CNN (AlexNet)

In [ ]:
#Find the Features from Alexnet's FC layer for n number of test images and we will get n x 4096
i=0
features_test=np.zeros(shape=(y_test.shape[0],4096))
for directory_path in glob.glob("/content/drive/MyDrive/BrainUltrasound_Multiclass/valid*"):
    for img_path in glob.glob(os.path.join(directory_path, "*.jpg")):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.resize(img, (227, 227))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        img = np.expand_dims(img, axis=0)
        FC_output = FC_layer_model.predict(img)
        features_test[i]=FC_output
        i+=1


In [ ]:
#Create DataFrame with features and coloumn name
test_features=pd.DataFrame(data=features_test,columns=feature_col)
feature_col = np.array(feature_col)

print('Test Features Shape:', test_features.shape)
print('Test Labels Shape:', test_label_ids.shape)


#### Classification by Random Forest

In [ ]:
#Feed the features of the test images to Random Forest Classifier to predict its class
predictions = rf.predict(test_features)
predictions

### Checking the Accuracy of the Novel Model

In [ ]:
accuracy=accuracy_score(predictions , test_label_ids)
print('Accuracy:', accuracy*100, '%.')

### Testing

In [ ]:

img_path="/content/BrainUltrasound_Multiclass/test/hydracenphay/Copy of hydracenphay_15j_aug_5_png.rf.12402a8a27124b6ddb048f0972f514ae.jpg"
img = cv2.imread(img_path, cv2.IMREAD_COLOR)
img = cv2.resize(img, (227, 227))
img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
plt.imshow(img)
img = np.expand_dims(img, axis=0)
FC_output = FC_layer_model.predict(img)
image_features=pd.DataFrame(data=FC_output,columns=feature_col)
predictions = rf.predict(image_features)
print("It's",id_to_label[predictions[0]])